## Minor corrections to the TrackPlant3D dataset labels

In [48]:
from pathlib import Path

import numpy as np

### 1. Mirrored points and labels in `187_tomato1_control1_plant2_D07.txt`

In [49]:
def rotate_180_y(points):
    """
    Rotate point cloud 180 degrees around Y-axis

    Args:
        points: numpy array of shape (N, 3) where each row is [x, y, z]

    Returns:
        rotated_points: numpy array of same shape with rotated coordinates
    """
    # 180 degree rotation matrix around Y-axis
    # cos(180°) = -1, sin(180°) = 0
    rotation_matrix = np.array([[-1, 0, 0], [0, 1, 0], [0, 0, -1]])

    # Apply rotation: multiply each point by the rotation matrix
    rotated_points = points @ rotation_matrix.T

    print(f"Rotated points by 180° in Y-axis")

    return rotated_points


def rotate_and_swap_labels(
    file_path: str | Path, label_map: dict, output_path: str | Path = None
):
    """Swap labels for wrongly labelled sequences according to label map, e.g. 1->2, 2->1, 3->4, 4->3"""

    points = np.loadtxt(file_path)

    # Rotate points 180 degrees around Y-axis
    points[:, :3] = rotate_180_y(points[:, :3])

    # Swap labels
    labels = points[:, 3].astype(int)
    new_labels = np.array([label_map.get(label, label) for label in labels])
    points[:, 3] = new_labels

    # Save to output path or overwrite original
    save_path = output_path if output_path else file_path
    np.savetxt(save_path, points, fmt="%f %f %f %d")
    print(f"Label changes: 1↔2, 3↔4")
    print(f"Saved file to: {save_path}")

    return points


##### Wrongly labelled scan 187_tomato1_control1_plant2_D07.txt #####
# Rotate 180° in Y-axis then swap labels (verified in CloudCompare that this would be the correct orientation and labels)
file_path = "../data/TrackPlant3D/gt/tomato/187_tomato1_control1_plant2_D07.txt"
output_path = (
    "../data/TrackPlant3D/corrections/gt_edited/tomato/187_tomato1_control1_plant2_D07.txt"
)
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

label_map = {1: 2, 2: 1, 3: 4, 4: 3}
rotate_and_swap_labels(file_path, label_map=label_map, output_path=output_path)


Rotated points by 180° in Y-axis
Label changes: 1↔2, 3↔4
Saved file to: ../data/TrackPlant3D/corrections/gt_edited/tomato/187_tomato1_control1_plant2_D07.txt


array([[ -3.279771,  17.77515 ,  13.14857 ,   4.      ],
       [ -3.890302,  17.772505,  13.099011,   4.      ],
       [ -3.492219,  17.971672,  13.164081,   4.      ],
       ...,
       [ -6.342129,  21.965334, -13.144647,   3.      ],
       [ -6.972039,  22.502869, -13.164081,   3.      ],
       [ -7.774089,  22.503471, -12.997791,   3.      ]], shape=(7961, 4))

### 2. Few wrongly labeled points in scan `231_tomato1_drought_plant3_D08`

The following points are labelled as stem (0) in leaf 1 but they're in the middle so they should clearly be leaf 1:

In [50]:
# Got these visually in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/231_tomato1_drought_plant3_D08_points_to_relabel_as_1.txt"
np.sort(np.loadtxt(file, delimiter=",")[...,0].astype(int))

array([ 15,  16,  17,  25,  26,  37,  55,  59,  60,  61,  62,  75,  76,
       106, 107, 109, 110, 116, 117, 129, 130, 158, 159, 162, 167, 168,
       179, 216, 217, 218, 221, 222, 230, 231, 282, 283, 288, 289])

In [51]:
indices = np.array([15, 16, 17, 25, 26, 37, 55, 59, 60, 61, 62, 75, 76, 106, 107, 109, 110, 116,
                    117, 129, 130, 158, 159, 162, 167, 168, 179, 216, 217, 218, 221, 222, 230, 231, 282, 283,
                    288, 289])

In [52]:
input_path = "../data/TrackPlant3D/gt/tomato/231_tomato1_drought_plant3_D08.txt"
output_path = "../data/TrackPlant3D/corrections/gt_edited/tomato/231_tomato1_drought_plant3_D08.txt"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [53]:
points = np.loadtxt(input_path)
points[indices, 3] = 1

np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points from label 0 to label 1")
print(f"Saved corrected file to: {output_path}")

Changed 38 points from label 0 to label 1
Saved corrected file to: ../data/TrackPlant3D/corrections/gt_edited/tomato/231_tomato1_drought_plant3_D08.txt


### 3. Some sparse points in stem labeled as leaves in `312_tomato1_shade_plant1_D06.txt`

In [54]:
## Points to relabel picked by visual inspection in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/312_tomato1_shade_plant1_D06_points_to_relabel_as_0.txt"
np.sort(np.loadtxt(file, delimiter=",")[...,0].astype(int))

array([ 733,  736,  933,  937,  941,  943,  945, 1279, 1284, 1286, 1692,
       1698, 1706, 1710, 2200, 2218, 2219, 2220, 2221, 2732, 2738, 2739,
       2740, 2742, 2744, 2745, 3255, 3256, 3591, 3592, 3593, 3594, 3595,
       3596, 3597, 3598, 3599, 3600, 3602, 3603, 3604, 3605, 3606, 3607,
       3876, 3877, 4112])

In [55]:
indices = np.array([ 733,  736,  933,  937,  941,  943,  945, 1279, 1284, 1286, 1692,
       1698, 1706, 1710, 2200, 2218, 2219, 2220, 2221, 2732, 2738, 2739,
       2740, 2742, 2744, 2745, 3255, 3256, 3591, 3592, 3593, 3594, 3595,
       3596, 3597, 3598, 3599, 3600, 3602, 3603, 3604, 3605, 3606, 3607,
       3876, 3877, 4112])

In [56]:
input_path = "../data/TrackPlant3D/gt/tomato/312_tomato1_shade_plant1_D06.txt"
output_path = "../data/TrackPlant3D/corrections/gt_edited/tomato/312_tomato1_shade_plant1_D06.txt"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [57]:
points = np.loadtxt(input_path)

# Relabel selected points as stem (0)
points[indices, 3] = 0

# Save the modified points
np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points from label 0 to label 1")
print(f"Saved corrected file to: {output_path}")

Changed 47 points from label 0 to label 1
Saved corrected file to: ../data/TrackPlant3D/corrections/gt_edited/tomato/312_tomato1_shade_plant1_D06.txt


### 4. Scan `128_tobacco_control_plant3_D08` leaf label 1 has two leaves

There is leaf 6 (if looking at the next scan) which is already clearly visible in this scan but still labeled as leaf 1. This is a mistake in the original dataset, so we can easily correct to gain an extra leaf label

In [58]:
## Points to relabel obtained from visual inspection in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/128_tobacco_control_plant3_D08_relabel_as_6.txt"
np.sort(np.loadtxt(file, delimiter=",")[...,0].astype(int))

array([2666, 2668, 2669, 2671, 2672, 2675, 2808, 2814, 2828, 2829, 2830,
       2831, 2832, 2833, 2834, 2835, 2974, 2975, 2977, 2978, 2980, 2981,
       2984, 2985, 2986, 2987, 2988, 2989, 2990, 2991, 2992, 2993, 2994,
       3112, 3116, 3117, 3118, 3121, 3122, 3123, 3126, 3127, 3129, 3130,
       3131, 3132, 3133, 3134, 3135, 3136, 3137, 3138, 3139, 3140, 3252,
       3257, 3258, 3259, 3260, 3262, 3263, 3264, 3267, 3268, 3269, 3271,
       3272, 3273, 3277, 3278, 3279, 3282, 3283, 3285, 3286, 3287, 3288,
       3289, 3290, 3408, 3411, 3412, 3415, 3416, 3418, 3419, 3420, 3422,
       3423, 3424, 3427, 3428, 3431, 3433, 3435, 3548, 3551, 3555, 3558])

In [59]:
indices = np.array([2666, 2668, 2669, 2671, 2672, 2675, 2808, 2814, 2828, 2829, 2830,
       2831, 2832, 2833, 2834, 2835, 2974, 2975, 2977, 2978, 2980, 2981,
       2984, 2985, 2986, 2987, 2988, 2989, 2990, 2991, 2992, 2993, 2994,
       3112, 3116, 3117, 3118, 3121, 3122, 3123, 3126, 3127, 3129, 3130,
       3131, 3132, 3133, 3134, 3135, 3136, 3137, 3138, 3139, 3140, 3252,
       3257, 3258, 3259, 3260, 3262, 3263, 3264, 3267, 3268, 3269, 3271,
       3272, 3273, 3277, 3278, 3279, 3282, 3283, 3285, 3286, 3287, 3288,
       3289, 3290, 3408, 3411, 3412, 3415, 3416, 3418, 3419, 3420, 3422,
       3423, 3424, 3427, 3428, 3431, 3433, 3435, 3548, 3551, 3555, 3558])

In [60]:
filename = "128_tobacco_control_plant3_D08.txt"
input_path = f"../data/TrackPlant3D/gt/tobacco/{filename}"
output_path = f"../data/TrackPlant3D/corrections/gt_edited/tobacco/{filename}"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [61]:
points = np.loadtxt(input_path)

# Relabel selected points
points[indices, 3] = 6

# Save the modified points
np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points from label 0 to label 6")
print(f"Saved corrected file to: {output_path}")

Changed 99 points from label 0 to label 6
Saved corrected file to: ../data/TrackPlant3D/corrections/gt_edited/tobacco/128_tobacco_control_plant3_D08.txt


### 5. Scan `134_tobacco_heat_plant2_D04` leaf 5 noisy points in peduncle of another leaf

In [62]:
## Points to relabel obtained from visual inspection in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/134_tobacco_heat_plant2_D04_relabel_as_0.txt"
np.sort(np.loadtxt(file, delimiter=",")[...,0].astype(int))

array([3224, 3364, 3365, 3366, 3367, 3370, 3506, 3507, 3508, 3509, 3510,
       3511, 3512, 3513, 3514, 3515, 3639, 3640, 3641, 3642, 3643, 3644,
       3645, 3646, 3647, 3785, 3786, 3787, 3788, 3789, 3790, 3791, 3792,
       3794, 3972, 3974, 3976, 3977, 3979, 3980, 3981, 3982, 3983, 3984,
       4224, 4225, 4227, 4235, 4241, 4248, 4466, 4471, 4477, 4673, 4676,
       5669, 5672, 5673, 5757, 5760])

In [63]:
indices = np.array([3224, 3364, 3365, 3366, 3367, 3370, 3506, 3507, 3508, 3509, 3510,
       3511, 3512, 3513, 3514, 3515, 3639, 3640, 3641, 3642, 3643, 3644,
       3645, 3646, 3647, 3785, 3786, 3787, 3788, 3789, 3790, 3791, 3792,
       3794, 3972, 3974, 3976, 3977, 3979, 3980, 3981, 3982, 3983, 3984,
       4224, 4225, 4227, 4235, 4241, 4248, 4466, 4471, 4477, 4673, 4676,
       5669, 5672, 5673, 5757, 5760])

In [64]:
filename = "134_tobacco_heat_plant2_D04.txt"
input_path = f"../data/TrackPlant3D/gt/tobacco/{filename}"
output_path = f"../data/TrackPlant3D/corrections/gt_edited/tobacco/{filename}"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [65]:
points = np.loadtxt(input_path)

# Relabel selected points
points[indices, 3] = 0

# Save the modified points
np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points to label 0")
print(f"Saved corrected file to: {output_path}")

Changed 60 points to label 0
Saved corrected file to: ../data/TrackPlant3D/corrections/gt_edited/tobacco/134_tobacco_heat_plant2_D04.txt


### 6. `141_tobacco_heat_plant3_D06.txt` in leaf 6 is currently labelled as stem (0)

In [66]:
## Points to relabel obtained from visual inspection in CloudCompare
file = "../data/TrackPlant3D/corrections/points_lists/141_tobacco_heat_plant3_D06_relabel_as_6.txt"
np.loadtxt(file, delimiter=",")[..., 0].astype(int)

array(6800)

In [67]:
indices = np.array([6800])

In [68]:
filename = "141_tobacco_heat_plant3_D06.txt"
input_path = f"../data/TrackPlant3D/gt/tobacco/{filename}"
output_path = f"../data/TrackPlant3D/corrections/gt_edited/tobacco/{filename}"
if not Path(output_path).parent.exists():
    Path(output_path).parent.mkdir(parents=True)

In [69]:
points = np.loadtxt(input_path)

# Relabel selected points
points[indices, 3] = 6

# Save the modified points
np.savetxt(output_path, points, fmt="%f %f %f %d")
print(f"Changed {len(indices)} points from label 5 to label 0")
print(f"Saved corrected file to: {output_path}")

Changed 1 points from label 5 to label 0
Saved corrected file to: ../data/TrackPlant3D/corrections/gt_edited/tobacco/141_tobacco_heat_plant3_D06.txt
